In [9]:
# ==========================
# Task 3: Preprocessing + Hypothesis Testing
# ==========================

import os
import pandas as pd
import numpy as np
from scipy import stats

# Step 1: Load raw DVC data
dvc_path = "../data/dvc_raw/raw_data.txt"
if not os.path.exists(dvc_path):
    raise FileNotFoundError(f"File not found at {dvc_path}")

df = pd.read_csv(dvc_path, sep="|")

# Step 2: Preprocess
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")
df.fillna({'citizenship': 'Unknown', 'totalpremium': 0, 'totalclaims': 0, 'gender': 'Unknown'}, inplace=True)

# Derived metrics
df['claim_occurred'] = np.where(df['totalclaims'] > 0, 1, 0)
df['claim_severity'] = df.apply(lambda x: x['totalclaims'] if x['claim_occurred'] == 1 else np.nan, axis=1)
df['margin'] = df['totalpremium'] - df['totalclaims']

# Step 3: H1 - Risk differences across provinces
provinces = df['province'].dropna().unique()
if len(provinces) >= 2:
    group_a = df[df['province'] == provinces[0]]['claim_occurred']
    group_b = df[df['province'] == provinces[1]]['claim_occurred']
    t_stat, p_val = stats.ttest_ind(group_a, group_b, nan_policy='omit')
    print(f"H1 Province: t-stat={t_stat:.3f}, p-value={p_val:.5f}")
    print("Reject H0" if p_val < 0.05 else "Fail to reject H0")
else:
    print("Not enough provinces to test H1")

# Step 4: H2 - Risk differences across zip codes
zip_codes = df['postalcode'].dropna().unique()
if len(zip_codes) >= 2:
    group_a = df[df['postalcode'] == zip_codes[0]]['claim_occurred']
    group_b = df[df['postalcode'] == zip_codes[1]]['claim_occurred']
    t_stat, p_val = stats.ttest_ind(group_a, group_b, nan_policy='omit')
    print(f"H2 Zip Code (Risk): t-stat={t_stat:.3f}, p-value={p_val:.5f}")
    print("Reject H0" if p_val < 0.05 else "Fail to reject H0")
else:
    print("Not enough zip codes to test H2")

# Step 5: H3 - Margin differences across zip codes
if len(zip_codes) >= 2:
    group_a = df[df['postalcode'] == zip_codes[0]]['margin']
    group_b = df[df['postalcode'] == zip_codes[1]]['margin']
    t_stat, p_val = stats.ttest_ind(group_a, group_b, nan_policy='omit')
    print(f"H3 Margin: t-stat={t_stat:.3f}, p-value={p_val:.5f}")
    print("Reject H0" if p_val < 0.05 else "Fail to reject H0")
else:
    print("Not enough zip codes to test H3")

# Step 6: H4 - Gender differences
genders = df['gender'].dropna().unique()
if len(genders) >= 2:
    group_a = df[df['gender'] == genders[0]]['claim_occurred']
    group_b = df[df['gender'] == genders[1]]['claim_occurred']
    t_stat, p_val = stats.ttest_ind(group_a, group_b, nan_policy='omit')
    print(f"H4 Gender: t-stat={t_stat:.3f}, p-value={p_val:.5f}")
    print("Reject H0" if p_val < 0.05 else "Fail to reject H0")
else:
    print("Not enough gender categories to test H4")


C:\Users\user\AppData\Local\Temp\ipykernel_6076\3229863881.py:15: DtypeWarning: Columns (32,37) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(dvc_path, sep="|")


H1 Province: t-stat=3.119, p-value=0.00181
Reject H0
H2 Zip Code (Risk): t-stat=nan, p-value=nan
Fail to reject H0
H3 Margin: t-stat=-0.486, p-value=0.62687
Fail to reject H0
H4 Gender: t-stat=2.440, p-value=0.01468
Reject H0
